# payment_risk_v1 exploration

Builds (or loads) a `payment_risk_v1` training set, splits it by time, trains on one GPU through the same TrainJob as the pipeline, evaluates on validation and saves the model. Every step is a function in `risk_model.steps`, shared with the `payment-risk-training` pipeline.

In [ ]:
import json
import os
import uuid

from risk_model import steps
from risk_model.data import DatasetReader
from risk_model.evaluate import GateEvaluator
from risk_model.split import WindowSplitter
from risk_model.train import RiskTrainer, load_booster, save_model, score

In [ ]:
windows = {
    "train": "2026-07-29..2026-09-30",
    "validation": "2026-10-01..2026-10-14",
    "test": "2026-10-15..2026-10-28",
}
cutoff = "2026-10-29T00:00:00Z"
run_id = f"notebook-{uuid.uuid4().hex[:8]}"
splitter = WindowSplitter(**windows)

## 1. Training set

Leave `snapshot` empty to build a new version through Feast, or paste a recorded snapshot ID to load it.

In [ ]:
snapshot = ""
snapshot = snapshot or steps.assemble_examples(**windows, cutoff=cutoff, run_id=run_id)["snapshot"]
reader = DatasetReader.at(snapshot)
print(json.dumps(reader.provenance(), indent=2))

In [ ]:
print(steps.check_examples(snapshot=snapshot, run_id=run_id))
print(steps.split_windows(snapshot=snapshot, **windows, run_id=run_id))

## 2. Counts, fraud rate and nulls per split

In [ ]:
splits = {name: reader.read(splitter[name]) for name in windows}
for name, split in splits.items():
    print(name, len(split.y), "sample fraud rate", float(split.y.mean()))
nulls = {
    feature: float((splits["train"].x[:, index] != splits["train"].x[:, index]).mean())
    for index, feature in enumerate(reader.features)
}
for feature, share in sorted(nulls.items(), key=lambda item: -item[1])[:15]:
    print(f"{share:6.1%}  {feature}")

## 3. Ablation of the derived request features

Each `request_features__*` column is removed in turn from a CPU model on a 2% negative sample. Columns whose removal does not lower validation PR-AUC stay out of `payment_risk_v1` before the first registered version.

In [ ]:
small = reader.read(splitter["train"], rate=0.02)
validation = splits["validation"]
evaluator = GateEvaluator()


def pr_auc(names):
    booster = RiskTrainer({"max_depth": 5}, device="cpu").fit(
        small.select(names), validation.select(names)
    )
    scores = {"model": score(booster, validation.select(names).x)}
    thresholds = {"model": evaluator.thresholds(validation.y, scores["model"], validation.w)}
    return evaluator.evaluate(thresholds, validation.y, validation.w, scores)["model"]["pr_auc"]


derived = [name for name in reader.features if name.startswith("request_features__")]
full = pr_auc(reader.features)
print("all features", round(full, 4))
for name in derived:
    print(f"without {name}", round(pr_auc([f for f in reader.features if f != name]), 4))

## 4. Single-GPU training

The same TrainJob as the pipeline's `fit_on_gpu` step. Notebooks cannot take GPU claims, so training runs there.

In [ ]:
fit = steps.fit_on_gpu(snapshot=snapshot, **windows, params="{}", run_id=run_id)
print(fit)

## 5. Validation and saving

In [ ]:
booster, order = load_booster(steps.download(fit["model_uri"].rstrip("/") + "/model.json"))
assert order == reader.features
scores = {"model": score(booster, validation.x)}
thresholds = {"model": evaluator.thresholds(validation.y, scores["model"], validation.w)}
print(thresholds["model"])
print(evaluator.evaluate(thresholds, validation.y, validation.w, scores)["model"])
save_model(booster, os.path.expanduser("~/models/payment_risk_v1"))